# Lab 2: Deploying a Machine Learning Model with FastAPI

## Learning Outcomes

- Build APIs using FastAPI.
- Validate input using Pydantic.
- Load a trained machine learning model.
- Create prediction endpoints.
- Test APIs using Requests.
- Use Swagger UI for API documentation.
- Compare Flask and FastAPI approaches.

In [1]:
# Install required packages
!pip install fastapi uvicorn scikit-learn pandas numpy joblib requests nest_asyncio

## Part 1: Import Libraries

In [2]:
# General-purpose libraries
import numpy as np
import pandas as pd
import joblib

# Machine learning libraries
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score

## Part 2: Load Dataset

In [3]:
# Load California Housing dataset
housing = fetch_california_housing(as_frame=True)

df = housing.frame

print(df.shape)
df.head()

(20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## Part 3: Prepare Features

In [4]:
# Select features used in Lab 1 for consistency
X = df[['AveRooms', 'HouseAge']]

# Target variable
y = df['MedHouseVal']

X.head()

,AveRooms,HouseAge
0,6.984127,41.0
1,6.238137,21.0
2,8.288136,52.0
3,5.817352,52.0
4,6.281853,52.0


## Part 4: Split Dataset

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

## Part 5: Train Model

In [6]:
# Train a Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("Training complete.")

Training complete.


## Part 6: Evaluate Model

In [7]:
# Make predictions on unseen data
predictions = model.predict(X_test)

# Calculate evaluation metrics
rmse = root_mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

RMSE: 1.1274
R² Score: 0.0301


## Part 7: Save the Model

In [8]:
# Save trained model
joblib.dump(model, "fastapi_model.joblib")

print("Model saved successfully.")

Model saved successfully.


## Part 8: Create FastAPI Application

In [9]:
from fastapi import FastAPI
from pydantic import BaseModel, Field

In [10]:
# Create application
app = FastAPI(
    title="Housing Prediction API",
    description="Predict house values using a machine learning model",
    version="1.0"
)

# Load saved model
model = joblib.load("fastapi_model.joblib")

## Part 9: Create Request Schema

In [11]:
class HousingRequest(BaseModel):

    rooms: float = Field(gt=0, description="Average rooms")
    age: float = Field(ge=0, description="House age")

## Part 10: Endpoints

In [12]:
@app.get("/")
def home():

    return {
        "message": "Housing Prediction API",
        "docs": "/docs",
        "health": "/health"
    }

In [13]:
@app.get("/health")
def health():

    return {
        "status": "healthy",
        "model_loaded": True
    }

In [14]:
@app.post("/predict")
def predict(data: HousingRequest):

    prediction = model.predict([[data.rooms, data.age]])

    return {
        "predicted_value": round(float(prediction[0]), 3)
    }

## Part 11: Run FastAPI

In [15]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

In [16]:
def run_server():

    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000
    )

In [17]:
threading.Thread(
    target=run_server,
    daemon=True
).start()

print("FastAPI started.")

FastAPI started.


## Part 12: Test the API

In [18]:
import requests

BASE_URL = "http://127.0.0.1:8000"

INFO:     Started server process [37492]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [19]:
# Test health endpoint
response = requests.get(f"{BASE_URL}/health")

print(response.status_code)
print(response.json())

INFO:     127.0.0.1:53361 - "GET /health HTTP/1.1" 200 OK
200
{'status': 'healthy', 'model_loaded': True}


In [20]:
# Test prediction endpoint
payload = {
    "rooms": 6,
    "age": 15
}

response = requests.post(
    f"{BASE_URL}/predict",
    json=payload
)

print(response.status_code)
print(response.json())

c:\Users\A.Shabut\Downloads\Session 7\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


INFO:     127.0.0.1:53362 - "POST /predict HTTP/1.1" 200 OK
200
{'predicted_value': 1.956}


## Swagger Documentation

Open:

http://127.0.0.1:8000/docs

Explore the automatically generated API documentation and test endpoints interactively.

# Reflection Questions

1. Why does FastAPI use Pydantic?
2. How does FastAPI differ from Flask?
3. What is Swagger UI used for?
4. Why is input validation important?
5. How could this API be deployed to Azure?